# GalaticBot: Production Simulation Workflow

Welcome to GalaticBot - a Monte Carlo simulator exploring the **Fermi Paradox** and the **Great Filter** hypothesis.

## What is the Fermi Paradox?

The Fermi Paradox asks: *"If intelligent civilizations are common in the galaxy, where is everybody?"*

This notebook provides a complete production workflow:
1. **Configure** simulation parameters
2. **Run** galactic civilization simulation
3. **Analyze** results and statistics
4. **Visualize** galaxy structure and civilization distribution
5. **Export** results for later analysis

---

## 1. Setup and Imports

Import the simulation widget from the `great_silence` library. All interactive components are provided by the library - no inline widget code needed!

In [ ]:
from great_silence.notebook import SimulationWidget, configure_notebook_display

# Configure display settings
configure_notebook_display()

## 2. Configure Simulation

The widget provides:
- **Preset selector**: Choose from scientifically-motivated scenarios
  - `moderate`: Balanced parameters (Fermi-consistent)
  - `optimistic`: Life is common, civilizations survive longer
  - `rare_earth`: Habitable planets are rare
  - `early_filter`: Great Filter in early stages (abiogenesis)
  - `late_filter`: Great Filter in late stages (self-destruction)
- **Parameter overrides**: Adjust stars, duration, random seed
- **Probe Expansion (Metallicity-Based Targeting)**:
  - Self-replicating probes need **metal-rich systems** for replication
  - Metallicity thresholds depend on technology level (Kardashev scale):
    - **K=0.85-0.95**: Need metal-rich systems ([Fe/H] > -0.3 default)
    - **K=0.95-1.20**: Can use solar-metallicity systems ([Fe/H] > -0.5)
    - **K>1.20**: Can utilize metal-poor systems ([Fe/H] > -1.0)
  - Probes prefer habitable planets but will target **any sufficiently metal-rich system**
- **Probe Sensors (Mid-Flight Course Correction)**:
  - Probes scan ahead with sensors (default 10 pc range)
  - Can **retarget to better planets** detected during flight:
    - Always prefer habitable over resource-only systems
    - Switch to higher metallicity targets (≥0.2 dex improvement)
  - Enables realistic opportunistic expansion
- **Monte Carlo mode**: Run multiple realizations for statistical analysis

### Usage:
1. Select a preset from the dropdown
2. (Optional) Adjust metallicity thresholds for different expansion strategies
3. (Optional) Modify sensor range to simulate different probe capabilities
4. (Optional) Override other parameters with sliders
5. Click **Run Simulation**
6. Watch the progress bar as the simulation executes

### Metallicity Targeting Examples:
- **Aggressive expansion**: Lower all metallicity thresholds (e.g., -0.5, -0.7, -1.2)
  - Probes can colonize more systems, faster expansion
- **Conservative expansion**: Raise thresholds (e.g., -0.1, -0.3, -0.5)
  - Only target metal-rich systems, slower but safer expansion
- **Technology-limited**: Keep early thresholds high, lower for advanced tech
  - Models breakthrough in resource extraction technology

In [ ]:
# Create simulation widget
widget = SimulationWidget()
# Display interactive interface
widget.display();  # Suppress output

---

**⚠️ Click the "Run Simulation" button above before proceeding**

---

## 3. Analyze Results

After the simulation completes, you can access detailed statistics and create custom analyses.

In [ ]:
# Access simulation results
if widget.results:
    # Get summary table
    summary = widget.runner.get_summary_table()
    print("Simulation Summary:")
    display(summary)
    
    # Get civilization dataframe for custom analysis
    civs_df = widget.runner.get_civilization_dataframe()
    print(f"\nTotal civilizations: {len(civs_df)}")
    
    if len(civs_df) > 0:
        print(f"Active: {civs_df['is_active'].sum()}")
        print(f"Extinct: {(~civs_df['is_active']).sum()}")
        print(f"\nFirst 5 civilizations:")
        display(civs_df.head())
else:
    print("No results yet. Run the simulation first!")

## 4. Visualize Galaxy Structure

Create an interactive 3D visualization of the galaxy with civilizations highlighted.

In [ ]:
if widget.runner:
    # Create interactive 3D plot with Plotly
    fig = widget.runner.plot_3d_galaxy(show_civilizations=True, backend='plotly')
    fig.show()
else:
    print("Run simulation first!")

## 5. Timeline Visualization

Show how civilizations emerge and go extinct over time.

In [ ]:
if widget.runner:
    # Timeline showing active vs total civilizations
    widget.runner.plot_timeline()
else:
    print("Run simulation first!")

## 6. Extinction Cause Analysis

Understand what killed civilizations - natural hazards or self-destruction?

In [ ]:
if widget.runner:
    # Pie chart of death causes
    widget.runner.plot_extinction_causes()
else:
    print("Run simulation first!")

## 7. Filter Civilizations

Filter civilizations by status, Kardashev level, or extinction cause.

In [ ]:
if widget.runner:
    # Example: Find all active Type I+ civilizations
    advanced_civs = widget.runner.filter_civilizations(
        status='active',
        min_kardashev=1.0
    )
    
    print(f"Advanced active civilizations (K >= 1.0): {len(advanced_civs)}")
    
    # Example: Find civilizations destroyed by supernovae
    supernova_deaths = widget.runner.filter_civilizations(
        status='extinct',
        death_cause='supernova'
    )
    
    print(f"Civilizations killed by supernovae: {len(supernova_deaths)}")
else:
    print("Run simulation first!")

## 8. Export Results

Save simulation results to HDF5 format for later analysis.

In [ ]:
if widget.runner:
    # Export to HDF5 (compressed)
    output_path = widget.export_results('output/my_simulation')
    print(f"Results saved to: {output_path}")
else:
    print("Run simulation first!")

## 9. Export Interactive Plots

Save interactive visualizations as standalone HTML files.

In [ ]:
if widget.runner:
    from great_silence.notebook import export_interactive_plot
    
    # Create and export 3D plot
    fig = widget.runner.plot_3d_galaxy(backend='plotly')
    html_path = export_interactive_plot(fig, 'output/galaxy_3d.html')
    
    print(f"Interactive 3D plot saved to: {html_path}")
    print("Open this file in a web browser to explore interactively!")
else:
    print("Run simulation first!")

---

## Next Steps

Explore other notebooks:
- **02_interactive_exploration.ipynb**: Load and analyze saved results
- **03_animation_generation.ipynb**: Create timeline animations and movies
- **04_expansion_trajectories_spheres.ipynb**: Visualize probe expansion patterns

### Advanced Usage: Custom Configuration

For more control, you can create custom configurations:

```python
from great_silence import SimulationConfig, GalaxySimulation
from great_silence.notebook import NotebookSimulationRunner

# Create custom config
config = SimulationConfig()
config.galaxy.total_stars = 200_000
config.civilization.fraction_develop_life = 0.5  # Optimistic!
config.simulation.simulation_duration_gyr = 13.8  # Age of universe

# Customize probe expansion parameters
config.civilization.metallicity_threshold_k085 = -0.2  # Early tech needs very rich systems
config.civilization.metallicity_threshold_k095 = -0.4  # Intermediate tech
config.civilization.metallicity_threshold_k120 = -0.8  # Advanced tech can use poorer systems

# Customize sensor capabilities
config.civilization.probe_sensor_range_pc = 20.0  # Extended sensor range
config.civilization.enable_mid_flight_retargeting = True  # Allow course corrections

# Run with custom config
runner = NotebookSimulationRunner(config, seed=123)
results = runner.run(verbose=True)
```

### Understanding Metallicity [Fe/H]

Metallicity measures heavy element abundance relative to the Sun:
- **[Fe/H] = 0.0**: Solar metallicity (our Sun)
- **[Fe/H] = -0.5**: Half solar metallicity (metal-poor but usable)
- **[Fe/H] = -1.0**: 1/10 solar metallicity (very metal-poor)
- **[Fe/H] = +0.3**: Double solar metallicity (metal-rich)

Self-replicating probes need metals for:
- Structures and hulls
- Electronics and computing
- Propulsion systems
- Manufacturing new probes

Lower-tech civilizations need abundant, easily-accessible metals. Advanced civilizations can extract trace elements from metal-poor systems.

---

**Questions about the Fermi Paradox?** This simulation explores:
- Drake Equation parameters
- Kardashev scale progression
- Great Filter hypotheses (early vs late)
- Astrophysical hazards (supernovae, GRBs)
- Self-destruction mechanisms
- **Metallicity-based expansion** (new!)
- **Sensor-guided probe navigation** (new!)
- Light-speed expansion constraints

Happy simulating! 🚀🌌